# 02 — Data Cleaning: Online Retail II

Notebook ini membersihkan data hasil profiling dan menghasilkan dataset siap EDA.

### Strategi cleaning
- Menyatukan nama kolom dengan format `snake_case`.
- Menormalkan tipe data.
- Menghapus exact duplicate.
- Memisahkan cancellation/return dari transaksi penjualan normal.
- Menghapus transaksi dengan `quantity <= 0` atau `unit_price <= 0` dari dataset penjualan.
- Mempertahankan missing `customer_id` agar analisis agregat transaksi tetap dapat dilakukan.
- Membuat fitur turunan seperti `revenue`, `year`, `month`, `day_name`, dan `hour`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Menentukan project root

In [3]:
from pathlib import Path

def find_project_root():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/home/jovyan/work"),
    ]

    # Hilangkan path duplikat
    candidates = list(dict.fromkeys(candidates))

    for base in candidates:
        if not base.exists():
            continue

        # Lokasi yang sesuai struktur project kamu
        raw_dir = base / "data" / "raw"

        if raw_dir.exists():
            files = [
                p for p in raw_dir.glob("online_retail_II*.xlsx")
                if not p.name.startswith("~$")
            ]

            if files:
                return base.resolve(), sorted(files)[0].resolve()

        # Fallback kalau dataset ternyata ada langsung di root
        files = [
            p for p in base.glob("online_retail_II*.xlsx")
            if not p.name.startswith("~$")
        ]

        if files:
            return base.resolve(), sorted(files)[0].resolve()

    raise FileNotFoundError(
        "Dataset online_retail_II*.xlsx tidak ditemukan di data/raw."
    )


ROOT, RAW_FILE = find_project_root()

INTERIM_DIR = ROOT / "data" / "interim"
OUTPUT_DIR = ROOT / "outputs" / "profiling"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Dataset      :", RAW_FILE)
print("Ukuran file :", f"{RAW_FILE.stat().st_size / 1024**2:,.2f} MiB")

Project root : /home/jovyan/work
Dataset      : /home/jovyan/work/data/raw/online_retail_II.xlsx
Ukuran file : 43.51 MiB


## 2. Load data

Notebook mencoba membaca cache dari notebook profiling terlebih dahulu. Jika cache belum ada, data akan dibaca langsung dari Excel.

In [4]:
raw_parquet = INTERIM_DIR / "online_retail_raw.parquet"
raw_csv = INTERIM_DIR / "online_retail_raw.csv"

if raw_parquet.exists():
    print("Membaca cache Parquet...")
    df = pd.read_parquet(raw_parquet)

elif raw_csv.exists():
    print("Membaca cache CSV...")
    df = pd.read_csv(raw_csv, low_memory=False, parse_dates=["InvoiceDate"])

else:
    excel_files = [
        p for p in ROOT.glob("online_retail_II*.xlsx")
        if not p.name.startswith("~$")
    ]
    if not excel_files:
        raise FileNotFoundError("File Online Retail II tidak ditemukan.")

    raw_file = sorted(excel_files)[0]
    print("Cache belum ada. Membaca Excel:", raw_file)

    xls = pd.ExcelFile(raw_file, engine="openpyxl")
    frames = []
    for sheet in xls.sheet_names:
        temp = pd.read_excel(raw_file, sheet_name=sheet, engine="openpyxl")
        temp["SourceYear"] = sheet
        frames.append(temp)

    df = pd.concat(frames, ignore_index=True)

print(f"Rows awal   : {len(df):,}")
print(f"Columns awal: {df.shape[1]}")
display(df.head())

Membaca cache Parquet...
Rows awal   : 1,067,371
Columns awal: 9


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010


## 3. Standardisasi nama kolom

In [5]:
rename_map = {
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
    "SourceYear": "source_year",
}

df = df.rename(columns=rename_map)

print(df.columns.tolist())

['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'source_year']


## 4. Normalisasi tipe data dan text fields

In [6]:
# String identifiers
for col in ["invoice", "stock_code", "description", "country", "source_year"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Numeric
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").astype("Int64")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")

# Datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

display(df.dtypes.to_frame("dtype"))

,dtype
invoice,string
stock_code,string
description,string
quantity,Int64
invoice_date,datetime64[us]
unit_price,float64
customer_id,Int64
country,string
source_year,string


## 5. Membuat quality flags sebelum filtering

In [7]:
df["is_cancelled"] = df["invoice"].str.startswith("C", na=False)
df["is_nonpositive_qty"] = df["quantity"].fillna(0) <= 0
df["is_nonpositive_price"] = df["unit_price"].fillna(0) <= 0

critical_cols = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "country",
]

df["has_missing_critical"] = df[critical_cols].isna().any(axis=1)

flags = pd.DataFrame({
    "condition": [
        "cancelled invoice",
        "quantity <= 0",
        "unit_price <= 0",
        "missing critical field",
        "missing customer_id",
        "exact duplicate",
    ],
    "rows": [
        df["is_cancelled"].sum(),
        df["is_nonpositive_qty"].sum(),
        df["is_nonpositive_price"].sum(),
        df["has_missing_critical"].sum(),
        df["customer_id"].isna().sum(),
        df.duplicated().sum(),
    ],
})

flags["pct"] = flags["rows"] / len(df) * 100
display(flags)

,condition,rows,pct
0,cancelled invoice,19494,1.83
1,quantity <= 0,22950,2.15
2,unit_price <= 0,6207,0.58
3,missing critical field,4382,0.41
4,missing customer_id,243007,22.77
5,exact duplicate,12133,1.14


## 6. Menghapus exact duplicate

Duplikat exact dihapus terlebih dahulu supaya satu baris transaksi tidak dihitung lebih dari sekali.

In [8]:
rows_before_dedup = len(df)
df = df.drop_duplicates().copy()
rows_after_dedup = len(df)

print(f"Rows sebelum dedup : {rows_before_dedup:,}")
print(f"Rows setelah dedup : {rows_after_dedup:,}")
print(f"Rows dihapus       : {rows_before_dedup - rows_after_dedup:,}")

Rows sebelum dedup : 1,067,371
Rows setelah dedup : 1,055,238
Rows dihapus       : 12,133


## 7. Memisahkan return/cancellation

Return/cancellation disimpan terpisah sehingga informasi tersebut tidak hilang. Dataset utama untuk EDA hanya memakai transaksi penjualan positif.

In [9]:
returns_mask = (
    df["is_cancelled"]
    | (df["quantity"] <= 0)
)

returns_df = df.loc[returns_mask].copy()

sales_mask = (
    ~df["is_cancelled"]
    & (df["quantity"] > 0)
    & (df["unit_price"] > 0)
    & ~df["has_missing_critical"]
)

clean_df = df.loc[sales_mask].copy()

print(f"Rows setelah dedup         : {len(df):,}")
print(f"Return/cancellation rows   : {len(returns_df):,}")
print(f"Clean positive sales rows  : {len(clean_df):,}")

Rows setelah dedup         : 1,055,238
Return/cancellation rows   : 22,890
Clean positive sales rows  : 1,029,609


## 8. Feature engineering

`revenue` dihitung sebagai:

\[
Revenue = Quantity \times UnitPrice
\]

Fitur waktu dibuat agar EDA lebih mudah.

In [10]:
clean_df["revenue"] = clean_df["quantity"].astype("float64") * clean_df["unit_price"]

clean_df["date"] = clean_df["invoice_date"].dt.date
clean_df["year"] = clean_df["invoice_date"].dt.year.astype("Int64")
clean_df["month"] = clean_df["invoice_date"].dt.to_period("M").astype("string")
clean_df["month_num"] = clean_df["invoice_date"].dt.month.astype("Int64")
clean_df["day_name"] = clean_df["invoice_date"].dt.day_name()
clean_df["hour"] = clean_df["invoice_date"].dt.hour.astype("Int64")

# Hapus flag teknis yang tidak diperlukan di dataset final
drop_flags = [
    "is_nonpositive_qty",
    "is_nonpositive_price",
    "has_missing_critical",
]

clean_df = clean_df.drop(columns=[c for c in drop_flags if c in clean_df.columns])

display(clean_df.head())

,invoice,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,source_year,is_cancelled,revenue,date,year,month,month_num,day_name,hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,False,83.40,2009-12-01,2009,2009-12,12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.00,2009-12-01,2009,2009-12,12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,False,81.00,2009-12-01,2009,2009-12,12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,False,100.80,2009-12-01,2009,2009-12,12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,False,30.00,2009-12-01,2009,2009-12,12,Tuesday,7


## 9. Validasi hasil cleaning

In [11]:
validation = pd.Series({
    "rows_clean": len(clean_df),
    "duplicate_rows": clean_df.duplicated().sum(),
    "cancelled_invoice_rows": clean_df["is_cancelled"].sum(),
    "quantity_le_0": (clean_df["quantity"] <= 0).sum(),
    "unit_price_le_0": (clean_df["unit_price"] <= 0).sum(),
    "missing_critical": clean_df[
        ["invoice", "stock_code", "description", "quantity", "invoice_date", "unit_price", "country"]
    ].isna().any(axis=1).sum(),
    "missing_customer_id": clean_df["customer_id"].isna().sum(),
}, name="value").to_frame()

display(validation)

,value
rows_clean,1029609
duplicate_rows,0
cancelled_invoice_rows,0
quantity_le_0,0
unit_price_le_0,0
missing_critical,0
missing_customer_id,236000


## 10. Perbandingan sebelum dan sesudah cleaning

In [12]:
cleaning_summary = pd.DataFrame({
    "stage": [
        "raw",
        "after_exact_dedup",
        "clean_positive_sales",
        "returns_or_cancellations",
    ],
    "rows": [
        rows_before_dedup,
        rows_after_dedup,
        len(clean_df),
        len(returns_df),
    ],
})

cleaning_summary["pct_of_raw"] = cleaning_summary["rows"] / rows_before_dedup * 100
display(cleaning_summary)

,stage,rows,pct_of_raw
0,raw,1067371,100.00
1,after_exact_dedup,1055238,98.86
2,clean_positive_sales,1029609,96.46
3,returns_or_cancellations,22890,2.14


## 11. Simpan dataset hasil cleaning

Format utama adalah **Parquet** karena lebih efisien untuk data besar. Jika engine Parquet tidak tersedia, notebook otomatis fallback ke CSV.

In [13]:
clean_parquet = PROCESSED_DIR / "online_retail_clean.parquet"
returns_parquet = PROCESSED_DIR / "online_retail_returns.parquet"

clean_csv = PROCESSED_DIR / "online_retail_clean.csv"
returns_csv = PROCESSED_DIR / "online_retail_returns.csv"

try:
    clean_df.to_parquet(clean_parquet, index=False)
    returns_df.to_parquet(returns_parquet, index=False)

    print("Clean dataset :", clean_parquet)
    print("Returns data  :", returns_parquet)

except Exception as e:
    print("Gagal menyimpan Parquet:", e)
    print("Fallback ke CSV...")

    clean_df.to_csv(clean_csv, index=False)
    returns_df.to_csv(returns_csv, index=False)

    print("Clean dataset :", clean_csv)
    print("Returns data  :", returns_csv)

cleaning_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
validation.reset_index(names="check").to_csv(
    OUTPUT_DIR / "validation_after_cleaning.csv",
    index=False
)

Clean dataset : /home/jovyan/work/data/processed/online_retail_clean.parquet
Returns data  : /home/jovyan/work/data/processed/online_retail_returns.parquet


## 12. Catatan metodologi

Keputusan cleaning yang digunakan:

- **Exact duplicate dihapus** karena dapat membuat transaksi dihitung ganda.
- **Cancellation/return dipisahkan**, bukan dihapus permanen.
- **Quantity dan unit price harus positif** pada dataset penjualan bersih.
- **Missing Customer ID tetap dipertahankan** untuk analisis omzet/produk/negara. Saat melakukan analisis customer, baris tanpa Customer ID akan difilter.
- Outlier harga atau quantity **tidak dihapus otomatis**, karena transaksi grosir dapat menghasilkan nilai yang memang besar. Outlier sebaiknya dianalisis secara kontekstual.